In [2]:
!pip install mlflow -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 132.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 113.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 97.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 133.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.2/216.2 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
# ============================================================
# 📊 Tabla Comparativa: Full Fine-Tuning vs LoRA vs QLoRA
# ============================================================
import mlflow
import pandas as pd
import json, os

# ── 1. Datos de la comparativa ──────────────────────────────
# Ajusta los valores reales de tus entrenamientos (LoRA y QLoRA)
comparison = {
    "Método": ["Full Fine-Tuning", "LoRA", "QLoRA"],

    # --- Configuración ---
    "Parámetros Entrenables": [
        "~3.2B (100%)",          # Full FT
        "~1.8B (~50%)",       # LoRA  ← ajusta con tu valor real
        "~24M (~0.4%)",       # QLoRA ← ajusta con tu valor real
    ],
    "Precisión": ["FP16/BF16", "BF16", "NF4 + BF16 adapters"],
    "VRAM Estimada (GB)": [">40", "~18-22", "~8-12"],

    # --- Resultados (sustituye con tus valores reales) ---
    "Training Loss Final": [
        0.74070,                 # Full FT
        0.03890,                  # LoRA
        0.04074,                  # QLoRA
    ],
    "Validation Loss Final": [
        0.843278,
        0.03840,                  # LoRA  ← tu valor real
        0.038170,                  # QLoRA ← pon tu valor real aquí
    ],

    # --- Recursos ---
    "Tamaño Adaptadores": [
        "~40 GB (modelo completo)",
        "~6 GB",                # ← ajusta
        "~3.07 GB",                # ← ajusta
    ],
    "Hardware Mínimo": [
        "A100 80GB / Multi-GPU",
        "A100 40GB",
        "T4 16GB / L4",
    ],
    "Riesgo de Overfitting": ["Alto", "Bajo", "Bajo"],

}

df = pd.DataFrame(comparison)

# ── 2. Mostrar en el notebook ────────────────────────────────
print("=" * 80)
print("📊 TABLA COMPARATIVA: Full Fine-Tuning vs LoRA vs QLoRA")
print("=" * 80)
display(df.set_index("Método").T)

# ── 3. Guardar como HTML ─────────────────────────────────────
html_table = df.set_index("Método").T.to_html(
    classes="table table-bordered table-striped",
    border=1
)

html_full = f"""
<html>
<head>
    <title>Comparativa Fine-Tuning</title>
    <style>
        body {{ font-family: Arial, sans-serif; padding: 20px; }}
        h1 {{ color: #2c3e50; }}
        table {{ border-collapse: collapse; width: 100%; margin-top: 15px; }}
        th, td {{ padding: 10px 14px; text-align: center; border: 1px solid #ddd; }}
        th {{ background-color: #3498db; color: white; }}
        tr:nth-child(even) {{ background-color: #f2f2f2; }}
        tr:hover {{ background-color: #e8f4f8; }}
    </style>
</head>
<body>
    <h1>📊 Comparativa: Full Fine-Tuning vs LoRA vs QLoRA</h1>
    <p><b>Modelo base:</b> Llama-3.2-3B &nbsp;|&nbsp; <b>Hardware:</b> A100 40GB (Colab)</p>
    {html_table}
</body>
</html>
"""

os.makedirs("artifacts", exist_ok=True)
html_path = "artifacts/comparativa_finetuning.html"
csv_path  = "artifacts/comparativa_finetuning.csv"
json_path = "artifacts/comparativa_finetuning.json"

with open(html_path, "w", encoding="utf-8") as f:
    f.write(html_full)
df.to_csv(csv_path, index=False)
df.to_json(json_path, orient="records", indent=2, force_ascii=False)

# ── 4. Registrar en MLflow ───────────────────────────────────
mlflow.set_experiment("Fase4_Comparativa_FineTuning")

with mlflow.start_run(run_name="comparativa_full_vs_lora_vs_qlora"):

    # Métricas numéricas (solo LoRA y QLoRA, Full FT no se ejecutó)
    #Full fine tunning
    mlflow.log_metric("FT_train_loss", 0.74070)      # ← tu valor real
    mlflow.log_metric("FT_val_loss", 0.843278)         # ← tu valor real
    mlflow.log_metric("FT_vram_gb", 40.0)            # ← estima o mide
    # LoRA
    mlflow.log_metric("lora_train_loss", 0.0389)      # ← tu valor real
    mlflow.log_metric("lora_val_loss", 0.0384)         # ← tu valor real
    mlflow.log_metric("lora_vram_gb", 6)            # ← estima o mide

    # QLoRA
    mlflow.log_metric("qlora_train_loss", 0.04074)      # ← tu valor real
    mlflow.log_metric("qlora_val_loss", 0.038170)        # ← tu valor real
    mlflow.log_metric("qlora_vram_gb", 3.07)           # ← estima o mide

    # Parámetros generales
    mlflow.log_param("modelo_base", "Llama-3.2-3B")
    mlflow.log_param("lora_rank", 16)
    mlflow.log_param("lora_alpha", 32)
    mlflow.log_param("qlora_quantization", "NF4")
    mlflow.log_param("full_ft_ejecutado", False)
    mlflow.log_param("hardware", "A100-40GB-Colab")

    # Artefactos
    mlflow.log_artifact(html_path)
    mlflow.log_artifact(csv_path)
    mlflow.log_artifact(json_path)

    # Tag descriptivo
    mlflow.set_tag("fase", "4 - Comparativa Final")
    mlflow.set_tag("descripcion", "Tabla comparativa Full FT vs LoRA vs QLoRA sobre Llama-3.2-3B")

    print("✅ Comparativa registrada en MLflow")
    print(f"   Run ID: {mlflow.active_run().info.run_id}")

print("\n📁 Artefactos guardados:")
print(f"   • {html_path}")
print(f"   • {csv_path}")
print(f"   • {json_path}")


📊 TABLA COMPARATIVA: Full Fine-Tuning vs LoRA vs QLoRA


Método,Full Fine-Tuning,LoRA,QLoRA
Parámetros Entrenables,~3.2B (100%),~1.8B (~50%),~24M (~0.4%)
Precisión,FP16/BF16,BF16,NF4 + BF16 adapters
VRAM Estimada (GB),>40,~18-22,~8-12
Training Loss Final,0.7407,0.0389,0.04074
Validation Loss Final,0.843278,0.0384,0.03817
Tamaño Adaptadores,~40 GB (modelo completo),~6 GB,~3.07 GB
Hardware Mínimo,A100 80GB / Multi-GPU,A100 40GB,T4 16GB / L4
Riesgo de Overfitting,Alto,Bajo,Bajo


2026/08/22 04:19:33 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/08/22 04:19:33 INFO mlflow.store.db.utils: Updating database tables
2026/08/22 04:19:35 INFO mlflow.tracking.fluent: Experiment with name 'Fase4_Comparativa_FineTuning' does not exist. Creating a new experiment.


✅ Comparativa registrada en MLflow
   Run ID: 0111c49776f34f818e55cdb4df356679

📁 Artefactos guardados:
   • artifacts/comparativa_finetuning.html
   • artifacts/comparativa_finetuning.csv
   • artifacts/comparativa_finetuning.json
